In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import poisson
from pathlib import Path

sns.set_theme(style='whitegrid')
PROCESSED = Path('../data/processed')

In [ ]:
df = pd.read_csv(PROCESSED / 'match_features.csv', parse_dates=['date'])

# only use competitive matches for attack/defence strength calculation
df = df[df['is_friendly'] == 0].copy()
print(f'competitive matches: {len(df)}')

In [ ]:
avg_home_goals = df['home_score'].mean()
avg_away_goals = df['away_score'].mean()
print(f'avg home goals: {avg_home_goals:.3f}')
print(f'avg away goals: {avg_away_goals:.3f}')

In [ ]:
# attack strength = how many goals a team scores vs the average
# defence strength = how many goals a team concedes vs the average
home_scored   = df.groupby('home_team')['home_score'].mean()
home_conceded = df.groupby('home_team')['away_score'].mean()
away_scored   = df.groupby('away_team')['away_score'].mean()
away_conceded = df.groupby('away_team')['home_score'].mean()

attack_home  = home_scored   / avg_home_goals
defence_home = home_conceded / avg_away_goals
attack_away  = away_scored   / avg_away_goals
defence_away = away_conceded / avg_home_goals

attack  = ((attack_home  + attack_away)  / 2).rename('attack')
defence = ((defence_home + defence_away) / 2).rename('defence')

team_strength = pd.DataFrame({'attack': attack, 'defence': defence})
print(team_strength.sort_values('attack', ascending=False).head(10))

In [ ]:
def predict_goals(home_team, away_team):
    if home_team not in team_strength.index or away_team not in team_strength.index:
        return None, None

    home_exp = (team_strength.loc[home_team, 'attack'] *
                team_strength.loc[away_team, 'defence'] *
                avg_home_goals)

    away_exp = (team_strength.loc[away_team, 'attack'] *
                team_strength.loc[home_team, 'defence'] *
                avg_away_goals)

    return home_exp, away_exp

home_exp, away_exp = predict_goals('Brazil', 'France')
print(f'Brazil vs France expected goals: {home_exp:.2f} - {away_exp:.2f}')

In [ ]:
def simulate_match(home_team, away_team, n=10000):
    home_exp, away_exp = predict_goals(home_team, away_team)
    if home_exp is None:
        return None

    home_goals = poisson.rvs(home_exp, size=n)
    away_goals = poisson.rvs(away_exp, size=n)

    return {
        'home_win': np.sum(home_goals > away_goals) / n,
        'draw':     np.sum(home_goals == away_goals) / n,
        'away_win': np.sum(home_goals < away_goals) / n
    }

result = simulate_match('Brazil', 'France')
print(f"Brazil win: {result['home_win']:.1%}")
print(f"Draw:       {result['draw']:.1%}")
print(f"France win: {result['away_win']:.1%}")

In [ ]:
def score_matrix(home_team, away_team, max_goals=5):
    home_exp, away_exp = predict_goals(home_team, away_team)
    matrix = np.zeros((max_goals+1, max_goals+1))
    for i in range(max_goals+1):
        for j in range(max_goals+1):
            matrix[i][j] = poisson.pmf(i, home_exp) * poisson.pmf(j, away_exp)
    return matrix

matrix = score_matrix('Brazil', 'France')

plt.figure(figsize=(7, 6))
sns.heatmap(matrix, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=range(6), yticklabels=range(6))
plt.xlabel('France goals')
plt.ylabel('Brazil goals')
plt.title('Scoreline probabilities: Brazil vs France')
plt.tight_layout()
plt.show()

In [ ]:
# backtest on historical matches
correct = 0
total   = 0

for _, row in df.iterrows():
    result = simulate_match(row['home_team'], row['away_team'], n=1000)
    if result is None:
        continue
    probs      = [result['home_win'], result['draw'], result['away_win']]
    prediction = [1, 0, -1][np.argmax(probs)]
    if prediction == row['result']:
        correct += 1
    total += 1

print(f'poisson accuracy: {correct/total:.1%} over {total} matches')

In [ ]:
team_strength.to_csv(PROCESSED / 'team_strength.csv')
pd.Series({'avg_home_goals': avg_home_goals,
           'avg_away_goals': avg_away_goals}).to_csv(PROCESSED / 'goal_averages.csv')
print('saved')